# 04 – Wind Radius / Core Size

Analyses the radial extent of 10 m winds for Hurricane Kirk across all
OIFS SST-perturbation experiments, following the wind_radius approach in
Ribberink et al. (2025).

**Method**: For each timestep, cut an E–W and N–S slice through the storm
centre (from the storm track) in the 10 m wind speed field. Plot the
wind speed contours in storm-relative coordinates. Key thresholds:
- 17 m/s (tropical storm force)
- 34 m/s (hurricane force)
- 50 m/s (~Category 3)

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from oifs_adapter import RUNS, OIFSRun
import hurricane_functions as hf

## 1. Load pre-computed tracks

In [ ]:
tracks = {}
for run_name in RUNS:
    safe_name = run_name.replace('+', 'p')
    path = f'../plots/tracks/track_{safe_name}.nc'
    if os.path.exists(path):
        tracks[run_name] = xr.open_dataset(path)
        print(f'Loaded: {run_name}')

## 2. Compute wind radius slices for each run

In [ ]:
wind_r = {}

for run_name, run_dir in RUNS.items():
    if run_name not in tracks:
        print(f'Skipping {run_name}: no track')
        continue
    print(f'Wind radius slices: {run_name}...')

    oifs_run = OIFSRun(run_dir)
    wind_ds = oifs_run.get_10m_wind()
    track_ds = tracks[run_name]

    # Compute wind speed magnitude
    ws = hf.magnitude(wind_ds['u10m'], wind_ds['v10m'])

    wind_r[run_name] = {}

    for j in range(len(track_ds.time)):
        lat_j = float(track_ds.lat[j])
        lon_j = float(track_ds.lon[j])

        # E-W slice (constant lat)
        r_EW = ws.isel(time=j).sel(lat=lat_j, method='nearest')
        r_EW = r_EW.assign_coords(lon=r_EW.lon - lon_j)  # storm-relative

        # N-S slice (constant lon)
        r_NS = ws.isel(time=j).sel(lon=lon_j, method='nearest')
        r_NS = r_NS.assign_coords(lat=r_NS.lat - lat_j)  # storm-relative

        if j == 0:
            wind_r[run_name]['EW'] = r_EW
            wind_r[run_name]['NS'] = r_NS
        else:
            wind_r[run_name]['EW'] = xr.concat(
                [wind_r[run_name]['EW'], r_EW], dim='time'
            )
            wind_r[run_name]['NS'] = xr.concat(
                [wind_r[run_name]['NS'], r_NS], dim='time'
            )

    print(f'  Done.')

## 3. Plot E-W and N-S wind radius for each run

In [ ]:
colors_line = {'baserun': '#1f77b4', '+3K': '#d62728', '+6K': '#ff7f0e', '-3K': '#2ca02c'}
wind_levels = [17, 34, 50]  # m/s thresholds

for direction in ['EW', 'NS']:
    coord = 'lon' if direction == 'EW' else 'lat'
    fig, ax = plt.subplots(figsize=(10, 5))

    for run_name, res in wind_r.items():
        da = res[direction]
        for thresh in wind_levels:
            cs = ax.contour(
                da.time.values,
                da[coord].values,
                da.values.T,
                levels=[thresh],
                colors=[colors_line.get(run_name, 'k')],
                linewidths=[1.5 if thresh == 17 else 1.0],
                linestyles=['-' if thresh == 17 else '--']
            )

    ax.set_xlabel('Date')
    ax.set_ylabel(f'Storm-relative {coord} (°)')
    ax.set_title(f'Hurricane Kirk – {direction} Wind Radius Slices\n'
                 f'(solid = 17 m/s, dashed = 34/50 m/s)')
    ax.set_ylim(-10, 10)
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

    plt.tight_layout()
    plt.savefig(f'../plots/kirk_windradius_{direction}.png', dpi=150)
    plt.show()
    print(f'Saved: ../plots/kirk_windradius_{direction}.png')

## 4. R17 radius time series (core size metric)

In [ ]:
# Compute R17 (radius of 17 m/s wind) for each timestep by finding max extent
# where wind speed >= 17 m/s in the E-W slice

fig, ax = plt.subplots(figsize=(10, 5))

for run_name, res in wind_r.items():
    ew = res['EW']
    r17_list = []
    for t in range(len(ew.time)):
        ws_slice = ew.isel(time=t)
        lons = ew.lon.values
        ws_vals = ws_slice.values
        # Find all lons where wind >= 17 m/s
        above = np.where(ws_vals >= 17)[0]
        if len(above) > 0:
            r17 = (lons[above.max()] - lons[above.min()]) / 2 * 111  # °→ km (approx)
        else:
            r17 = 0.0
        r17_list.append(r17)

    ax.plot(ew.time.values, r17_list, '.-',
            color=colors_line.get(run_name, 'k'), label=run_name)

ax.set_xlabel('Date')
ax.set_ylabel('R17 (km, from E-W slice)')
ax.set_title('Hurricane Kirk – Tropical Storm Wind Radius (R17)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('../plots/kirk_R17.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_R17.png')